# Virtual Try-On with `opentryon`

This notebook demonstrates the `opentryon` package's `vton` service directly in Python -- no CLI, server, or frontend required.

`opentryon` ships four virtual try-on adapters, all with the same `person_image` + `garment_image` -> `images` shape:

| Model id | Provider | Env var required |
|---|---|---|
| `flux-vto` | Black Forest Labs FLUX VTO | `BFL_API_KEY` |
| `nova-canvas` | Amazon Nova Canvas | `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY` |
| `kling-ai` | Kling AI (Kolors Virtual Try-On) | `KLING_AI_API_KEY` / `KLING_AI_SECRET_KEY` |
| `segmind` | Segmind Try-On Diffusion | `SEGMIND_API_KEY` |

We call all of them through the same entry point the CLI and MCP server use internally: `tryon.cli.runner.invoke_model()`. It always returns a structured `{"success": ...}` dict instead of raising, and supports `dry_run=True` to preview the resolved adapter call with **no API cost and no key required** -- which is what this notebook runs by default so it's executable out of the box.

Set `DRY_RUN = False` in the cells below (and export the relevant API key first) to actually generate an image.

## Setup

If you're running this from a fresh checkout, install `opentryon` first (from the repo root):

```bash
pip install -e .
```

Then load your API keys from `.env` (copy `env.template` -> `.env` and fill in the key for whichever model(s) you want to actually call).

In [ ]:
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

from tryon.cli.runner import invoke_model
from tryon.cli.registry import get_service

# Toggle this to False (and export the matching API key) to make a real call.
DRY_RUN = True

PERSON_IMAGE = "assets/person.jpg"
GARMENT_IMAGE = "assets/garment.jpg"
OUTPUT_DIR = Path("outputs/notebook_vton")

print("Available vton models:", list(get_service("vton").keys()))

## Run a single model

`invoke_model(service, model, **kwargs)` mirrors the CLI's `opentryon run vton <model> ...` and the MCP server's generated tools -- same arguments, same result shape.

In [ ]:
result = invoke_model(
    "vton",
    "flux-vto",
    person=PERSON_IMAGE,
    garment=GARMENT_IMAGE,
    garment_description="a red cotton t-shirt",
    output_dir=str(OUTPUT_DIR),
    dry_run=DRY_RUN,
)

result

## Render the result

When `dry_run=False`, `result["output_paths"]` holds the generated image(s) on disk, and `result["images_base64"]` holds the same images base64-encoded (handy for remote/MCP clients with no filesystem access). This cell displays them inline if the call actually ran.

In [ ]:
from base64 import b64decode
from io import BytesIO

from IPython.display import display
from PIL import Image

if result.get("success") and result.get("images_base64"):
    for b64 in result["images_base64"]:
        display(Image.open(BytesIO(b64decode(b64))))
elif result.get("dry_run"):
    print("Dry run only -- no image was generated. Set DRY_RUN = False to render a result here.")
else:
    print("No image returned:", result.get("error"))

## Compare all four models

Each `vton` adapter takes slightly different argument names for the same two images. This cell loops over all of them in dry-run mode so you can see the resolved call for each without needing every API key configured.

In [ ]:
# Per-model argument mapping -- see tryon/cli/registry.py for the full spec.
MODEL_ARGS = {
    "flux-vto": {"person": PERSON_IMAGE, "garment": GARMENT_IMAGE, "garment_description": "a red cotton t-shirt"},
    "nova-canvas": {"source_image": PERSON_IMAGE, "reference_image": GARMENT_IMAGE},
    "kling-ai": {"source_image": PERSON_IMAGE, "reference_image": GARMENT_IMAGE},
    "segmind": {"model_image": PERSON_IMAGE, "cloth_image": GARMENT_IMAGE},
}

for model, kwargs in MODEL_ARGS.items():
    result = invoke_model("vton", model, output_dir=str(OUTPUT_DIR), dry_run=True, **kwargs)
    print(f"{model}: success={result.get('success')} call={result.get('call')}")